# 01 · Oxford Flowers-102: Dataset Exploration

**SIBOL: System for Image-Based Observation and Learning**

This notebook prepares the dataset that all three approaches (HOG + Linear SVM, custom CNN, ResNet-18) share, verifies it, and explores it.

### How to run in a fresh Google Colab session
1. Make sure the three source files are in Google Drive at `MyDrive/sibol/data/`:
   `102flowers.tgz`, `imagelabels.mat`, `setid.mat`.
2. Make sure the SIBOL code is reachable: pushed to the GitHub repository in `REPO_URL`, **or** copied to Drive (then set `CODE_DIR_COLAB` to that folder).
3. **Runtime → Run all.** A GPU is not needed for this notebook.

### What happens
| Step | Where |
|---|---|
| Source files are read from | Google Drive (persistent) |
| Images are extracted to | Colab temporary disk `/content/flowers102` (fast, wiped when the runtime ends) |
| Figures are saved to | Google Drive `MyDrive/sibol/outputs/figures` (persistent) |

### Network policy
With `DOWNLOAD = False` (the default) this notebook **never downloads dataset files**. If a source file is missing it stops with a `FileNotFoundError` that names the missing files and where to put them. Only `DOWNLOAD = True` allows fetching missing files from the official site.

Dataset: Nilsback & Zisserman, *Automated Flower Classification over a Large Number of Classes*, ICVGIP 2008. [Project page](https://www.robots.ox.ac.uk/~vgg/data/flowers/102/).

## 1. Configuration

**All paths and options are set here.** Later cells use only these values.

In [ ]:
from pathlib import Path

# ---- SIBOL code (Colab) ------------------------------------------------------
REPO_URL = "https://github.com/didyouseejay/sibol.git"
CODE_DIR_COLAB = Path("/content/sibol")  # clone target; or a Drive copy, e.g. Path("/content/drive/MyDrive/sibol_code")

# ---- Google Drive (Colab) ----------------------------------------------------
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "sibol"
SOURCE_DIR_COLAB = DRIVE_PROJECT_DIR / "data"                # the three source files
FIGURES_DIR_COLAB = DRIVE_PROJECT_DIR / "outputs" / "figures"  # persistent figures

# ---- Colab temporary storage (fast local disk, wiped when the runtime ends) ----
WORK_ROOT_COLAB = Path("/content/flowers102")

# ---- Local runs (relative to the project root; all git-ignored except figures) ----
SOURCE_SUBDIR_LOCAL = "data"            # <project>/data/102flowers.tgz, ...
WORK_SUBDIR_LOCAL = "data/extracted"    # <project>/data/extracted/flowers-102/jpg/...
FIGURES_SUBDIR_LOCAL = "outputs/figures"

# ---- Behaviour -------------------------------------------------------------------
DOWNLOAD = False      # True = allow downloading MISSING source files from the official site
VERIFY_MD5 = True     # check source files against the official MD5 checksums (reads the whole 345 MB archive once)
SEED = 42             # makes random samples reproducible
N_SIZE_SAMPLES = 300  # images sampled for the size statistics
CLASS_ID = 0          # class shown in the single-class view

## 2. Runtime and Google Drive

Detects Colab and mounts Google Drive explicitly. Locally, this step is skipped.

In [ ]:
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    if not DRIVE_PROJECT_DIR.is_dir():
        raise FileNotFoundError(
            f"Google Drive is mounted, but the project folder {DRIVE_PROJECT_DIR} does not exist. "
            "Create it (or run notebooks/00_environment_setup.ipynb) and put the dataset files in its 'data' folder."
        )
    print(f"Google Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Running locally: Google Drive mount skipped.")

## 3. Load the SIBOL project code

In Colab the repository is cloned into temporary storage if it is not already there. The notebook then imports `src/data/flowers102.py` and checks that it is the current project version, not an older copy left from a previous run.

In [ ]:
import subprocess
import sys

HELPER = Path("src") / "data" / "flowers102.py"

if IN_COLAB:
    PROJECT_ROOT = CODE_DIR_COLAB
    if not (PROJECT_ROOT / HELPER).is_file():
        if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
            raise FileNotFoundError(
                f"{PROJECT_ROOT} exists but does not contain {HELPER}. It may be an empty clone or an old copy. "
                f"Delete it (e.g. run: !rm -rf {PROJECT_ROOT}) and re-run this cell, or set CODE_DIR_COLAB."
            )
        print(f"Cloning {REPO_URL} into {PROJECT_ROOT} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    # Walk upwards from the notebook's folder until we find the folder that contains src/.
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / HELPER).is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / HELPER).is_file():
    raise FileNotFoundError(
        f"SIBOL code not found: {PROJECT_ROOT / HELPER} is missing. If the GitHub repository is still empty, "
        "push the project first, or copy the project folder to Google Drive and set CODE_DIR_COLAB to it."
    )

# Forget any previously imported 'src' package so we import the files on disk now.
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import flowers102 as f102

if not hasattr(f102, "prepare_dataset"):
    raise ImportError(f"{f102.__file__} is an outdated helper (no prepare_dataset). Update the project code.")

commit = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"Project root : {PROJECT_ROOT}")
print(f"Helper module: {f102.__file__}")
print(f"Git commit   : {commit or 'n/a (not a git checkout)'}")

## 4. Resolve paths

In [ ]:
if IN_COLAB:
    SOURCE_DIR, WORK_ROOT, FIGURES_DIR = SOURCE_DIR_COLAB, WORK_ROOT_COLAB, FIGURES_DIR_COLAB
else:
    SOURCE_DIR = PROJECT_ROOT / SOURCE_SUBDIR_LOCAL
    WORK_ROOT = PROJECT_ROOT / WORK_SUBDIR_LOCAL
    FIGURES_DIR = PROJECT_ROOT / FIGURES_SUBDIR_LOCAL

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Source files   : {SOURCE_DIR}")
print(f"Extracted data : {WORK_ROOT / f102.TORCHVISION_SUBDIR}")
print(f"Figures        : {FIGURES_DIR}")
print(f"DOWNLOAD       : {DOWNLOAD}")

## 5. Verify the three source files

| File | Content |
|---|---|
| `102flowers.tgz` | the 8,189 images |
| `imagelabels.mat` | class label of every image |
| `setid.mat` | official train / val / test split |

With `DOWNLOAD = False`, a missing file stops the notebook here with instructions, and nothing is downloaded.

In [ ]:
if DOWNLOAD:
    fetched = f102.download_source_files(SOURCE_DIR)
    print(f"Downloaded: {fetched or 'nothing (all files already present)'}")
else:
    missing = f102.missing_source_files(SOURCE_DIR)
    if missing:
        raise FileNotFoundError(f102.missing_files_message(SOURCE_DIR, missing))

if VERIFY_MD5:
    print("Checking MD5 checksums (reading the 345 MB archive from Drive may take a few minutes)...")
source_report = f102.verify_source_files(SOURCE_DIR, check_md5=VERIFY_MD5)

for name, info in source_report.items():
    md5 = {True: "MD5 OK", False: "MD5 MISMATCH", None: "MD5 not checked"}[info["md5_ok"]]
    print(f"  {name:<16} {info['size']:>12,} bytes   {md5}")

## 6. Extract the images to temporary storage

Images are extracted from the archive on Drive to the fast local disk. If a complete extraction already exists in this runtime (all 8,189 images), it is reused. The two `.mat` files are copied next to the images, where torchvision expects them.

In [ ]:
import time

reused = f102.is_prepared(WORK_ROOT)
t0 = time.time()
f102.prepare_dataset(SOURCE_DIR, WORK_ROOT, download=False, check_md5=False)  # already verified above
elapsed = time.time() - t0

print("Reused existing extraction." if reused else f"Extracted images in {elapsed:.0f} s.")
print(f"Images available: {f102.count_images(WORK_ROOT):,} (expected {f102.EXPECTED_TOTAL_IMAGES:,})")

## 7. Load labels and the official splits

`torchvision.datasets.Flowers102` reads labels from `imagelabels.mat` and the official split from `setid.mat`. Loading never downloads anything.

In [ ]:
datasets = {split: f102.load_split(WORK_ROOT, split) for split in f102.SPLITS}
for split, ds in datasets.items():
    print(f"Loaded {split:<5}: {len(ds):,} images")

labels = {s: f102.get_labels(ds) for s, ds in datasets.items()}
files = {s: f102.get_image_files(ds) for s, ds in datasets.items()}

# Class names: newer torchvision versions provide them; otherwise use numbered names.
CLASS_NAMES = getattr(datasets["train"], "classes", None) or [f"class_{i:03d}" for i in range(f102.NUM_CLASSES)]

## 8. The 10 integrity checks

| # | Check | Expected |
|---|---|---|
| 1-3 | Split sizes | train 1020, val 1020, test 6149 |
| 4 | Total images | 8189 |
| 5-7 | Classes in each split | 102 |
| 8-9 | Images per class in train and val | exactly 10 |
| 10 | Splits are disjoint | no shared images |

If any check fails, the notebook stops so that nothing downstream uses bad data.

In [ ]:
checks = f102.integrity_checks(labels, files)
assert len(checks) == 10, f"expected 10 checks, got {len(checks)}"

for i, (name, passed, detail) in enumerate(checks, start=1):
    print(f"{i:>2}. [{'PASS' if passed else 'FAIL'}] {name}: {detail}")

n_passed = sum(passed for _, passed, _ in checks)
print(f"\n{n_passed}/10 checks passed.")
if n_passed != 10:
    raise RuntimeError("Integrity checks failed: investigate before using this data.")

## 9. Class distribution

Train and val are balanced by design (10 per class). The test split holds all remaining images, so it is **imbalanced**. That matters when we report accuracy later (we should also look at per-class / macro-averaged metrics).

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
for ax, s in zip(axes, f102.SPLITS):
    counts = np.array(f102.class_counts(labels[s]))
    ax.bar(np.arange(f102.NUM_CLASSES), counts, width=0.85)
    ax.set_title(f"{s}: {counts.sum()} images | min {counts.min()}, "
                 f"max {counts.max()}, mean {counts.mean():.1f} per class")
    ax.set_ylabel("images")
axes[-1].set_xlabel("class index (0-101)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "class_distribution.png", dpi=120)
plt.show()

# Most and least frequent classes in the test split.
test_counts = np.array(f102.class_counts(labels["test"]))
smallest = [CLASS_NAMES[i] for i in np.flatnonzero(test_counts == test_counts.min())]
print(f"Smallest test classes ({test_counts.min()} images each, {len(smallest)} classes): {smallest}")
order = np.argsort(-test_counts, kind="stable")
print("Largest test classes:", [(CLASS_NAMES[i], int(test_counts[i])) for i in order[:5]])

## 10. Sample images

A random grid from the training split (fixed seed, so the grid is reproducible).

In [ ]:
rng = random.Random(SEED)
idx = rng.sample(range(len(datasets["train"])), 16)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, i in zip(axes.flat, idx):
    image, label = datasets["train"][i]  # PIL image (no transform applied)
    ax.imshow(image)
    ax.set_title(f"{label}: {CLASS_NAMES[label]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Random training samples", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sample_images.png", dpi=120)
plt.show()

### 10b. All training images of one class

Useful for seeing intra-class variation (pose, scale, colour, background). Change `CLASS_ID` in the configuration cell to look at other classes.

In [ ]:
members = [i for i, y in enumerate(labels["train"]) if y == CLASS_ID]
fig, axes = plt.subplots(1, len(members), figsize=(2.2 * len(members), 2.6))
for ax, i in zip(np.atleast_1d(axes), members):
    ax.imshow(datasets["train"][i][0])
    ax.axis("off")
fig.suptitle(f"Class {CLASS_ID}: {CLASS_NAMES[CLASS_ID]} (all train images)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"class_{CLASS_ID:03d}_examples.png", dpi=120)
plt.show()

## 11. Image size statistics

Flowers-102 images have varying sizes. We check a random subset so we can choose sensible input sizes for HOG (fixed size needed) and the CNNs.

In [ ]:
rng = random.Random(SEED)
split_ds = datasets["train"]
pick = rng.sample(range(len(split_ds)), min(N_SIZE_SAMPLES, len(split_ds)))
sizes = np.array([split_ds[i][0].size for i in pick])  # (width, height)
widths, heights = sizes[:, 0], sizes[:, 1]
aspect = widths / heights

print(f"Sampled {len(pick)} training images")
print(f"Width : min {widths.min()}, median {np.median(widths):.0f}, max {widths.max()}")
print(f"Height: min {heights.min()}, median {np.median(heights):.0f}, max {heights.max()}")
print(f"Aspect ratio (w/h): min {aspect.min():.2f}, median {np.median(aspect):.2f}, max {aspect.max():.2f}")
print(f"Shorter side = 500 px in {np.mean(np.minimum(widths, heights) == 500):.0%} of sampled images")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(widths, heights, s=8, alpha=0.6)
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("Image sizes (random training subset)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "image_sizes.png", dpi=120)
plt.show()

## 12. Run summary

In [ ]:
print(f"Integrity checks : {n_passed}/10 passed")
print(f"Images           : {sum(len(d) for d in datasets.values()):,}")
print(f"Figures saved to : {FIGURES_DIR}")
for p in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  - {p.name}")

## 13. Observations

*Recorded from the 2026-11-11 Colab run (an earlier version of this notebook, using the same official files and the same 10 checks; archived in `notebooks/archive/`). After re-running this version, confirm the outputs above still match and update this section if anything differs.*

**Split verification**
- All 10 checks passed: train 1020, val 1020, test 6149 (total 8189); 102 classes in every split; exactly 10 images per class in train and val; 0 image files shared between splits.
- The official splits loaded by `torchvision.datasets.Flowers102` match the dataset authors' documentation, so they can be used as-is for all three approaches.

**Class balance**
- Train and val are perfectly balanced (10 images per class).
- Test is imbalanced: min 20, max 238, mean 60.3 images per class (about 12x between the smallest and largest class).
- Smallest test classes have 20 images each, including pink primrose, canterbury bells, moon orchid, fire lily and prince of wales feathers (the 2026-11-11 run listed only 5 of the tied classes).
- Most test images: petunia (238), passion flower (231), wallflower (176), water lily (174), watercress (164).
- *Implication:* overall test accuracy is weighted towards the large classes, so we should also report macro-averaged and per-class metrics.

**Visual characteristics** (from the 16-image random grid and the 10 pink primrose training images)
- Backgrounds are mostly natural and cluttered (leaves, grass, soil, fences), not plain studio backgrounds.
- Some images contain several flowers, or flowers alongside buds (e.g. artichoke, pink primrose).
- Colour varies within a class: the three morning glory samples are pink, white/lilac and blue; pink primrose ranges from almost white to deep pink.
- Scale and framing vary: some flowers fill the frame (corn poppy, colt's foot), others are small within the scene (cautleya spicata, fritillary).
- Several images carry photographer watermarks or copyright text (e.g. corn poppy, yellow iris, tree poppy, one pink primrose image).

**Image sizes** (random sample of 300 training images, seed 42)
- Width 500 to 857 px (median 667); height 500 to 861 px (median 500); aspect ratio (w/h) 0.58 to 1.71 (median 1.33).
- The scatter plot shows every sampled image has its shorter side equal to 500 px: landscape images are 500 px tall, portrait images are 500 px wide.
- *Implication:* images are not a fixed size, so each pipeline needs a resize step (e.g. resize the shorter side, then crop). A centre crop will cut off the edges of the most elongated images (aspect 0.58 and 1.71).
- *Limitation:* sizes were measured on 300 training images only, not on val or test.